# Swisstopo Enrichment direkt aus Postgres

Dieses Notebook:

1. verbindet sich mit deiner Neon/Postgres-Datenbank über `DATABASE_URL`
2. liest die Adressen aus `listing_details.address`
3. geocodiert jede eindeutige Adresse via GeoAdmin `SearchServer` nach LV95 (East/North)
4. holt pro Punkt: **ÖV-Score**, **Solar-Einstrahlung**, **Höhe ü.M.** und **Bevölkerung** (Hektarraster, nächstgelegene Zelle)
5. schreibt die Werte in `public.swisstopo_details` mit `ON CONFLICT (listing_id) DO UPDATE`

Bevor der grosse Loop läuft, gibt's eine **Einzeltest-Zelle**, damit du die Helper auf genau einer Adresse probieren kannst.

Optional am Ende: Diagnose-Zellen für NULL-Quoten und Constraints.


In [ ]:
%pip install pandas requests sqlalchemy psycopg2-binary tqdm -q

In [2]:

import os
import math
import time
from typing import Any, Dict, Optional, Tuple

import pandas as pd
import requests
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

print("DB_URL geladen:", DB_URL.split("@")[-1])


DB_URL geladen: ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require


## Konfiguration

In [3]:

SCHEMA = "public"
LISTING_DETAILS_TABLE = "listing_details"
TARGET_TABLE = "swisstopo_details"

ADDRESS_COLUMN = "address"
LISTING_ID_COLUMN = "listing_id"

REQUESTS_PER_SECOND = 5
DELAY = 1.0 / REQUESTS_PER_SECOND

API_SEARCH_URL   = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
API_IDENTIFY_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/identify"
API_HEIGHT_URL   = "https://api3.geo.admin.ch/rest/services/height"

IDENTIFY_LAYERS = "all:" + ",".join([
    "ch.are.erreichbarkeit-oev",
    "ch.bfe.solarenergie-eignung-daecher",
])
POP_LAYER = "all:ch.bfs.volkszaehlung-bevoelkerungsstatistik_einwohner"

print(f"Zieltable  : {SCHEMA}.{TARGET_TABLE}")
print(f"Quelltable : {SCHEMA}.{LISTING_DETAILS_TABLE}")
print(f"Layers     : {IDENTIFY_LAYERS}")
print(f"Rate limit : {REQUESTS_PER_SECOND} req/s  (delay {DELAY:.2f}s)")


Zieltable  : public.swisstopo_details
Quelltable : public.listing_details
Layers     : all:ch.are.erreichbarkeit-oev,ch.bfe.solarenergie-eignung-daecher
Rate limit : 5 req/s  (delay 0.20s)


## Datenbankverbindung

In [4]:

engine = create_engine(DB_URL, pool_pre_ping=True)

with engine.connect() as conn:
    version = conn.execute(text("select version()")).scalar()
    print("✅ Verbunden")
    print(version[:120] + "...")


✅ Verbunden
PostgreSQL 17.8 (130b160) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit...


## Listings mit Adresse laden

Gelesen wird direkt aus `listing_details.address`.


In [ ]:

query = text(f"""
    SELECT
        ld.{LISTING_ID_COLUMN} AS listing_id,
        ld.{ADDRESS_COLUMN}    AS address
    FROM {SCHEMA}.{LISTING_DETAILS_TABLE} ld
    WHERE ld.{ADDRESS_COLUMN} IS NOT NULL
      AND btrim(ld.{ADDRESS_COLUMN}) <> ''
    ORDER BY ld.{LISTING_ID_COLUMN}
""")

listing_df = pd.read_sql(query, engine)

print("Geladene Zeilen:", len(listing_df))
display(listing_df.head(10))


In [ ]:

import re

def normalize_address(addr: Any) -> str:
    if not isinstance(addr, str):
        return ""
    return re.sub(r"\s+", " ", addr.strip())

listing_df["address_norm"] = listing_df["address"].apply(normalize_address)
listing_df = listing_df[listing_df["address_norm"] != ""].copy()

print("Nach Normalisierung:", len(listing_df))
display(listing_df.head(10))


## GeoAdmin Helper

- `safe_num` / `safe_int`: robustes Zahlen-Parsing (None / NaN / leere Strings).
- `_request` / `_get_json`: HTTP-Wrapper mit einem Retry bei Connection-Fehlern.
- `geocode`: Adresse → LV95 (East/North). Achtung: SearchServer liefert `y = East`, `x = North`.
- `get_elevation`: Höhe ü.M. über den `height`-Endpoint.
- `identify` + `parse_identify`: **ÖV-Score & Solar** – mit `tolerance=1` nur die Zone/Dachfläche **am Punkt**, und pro Layer der **erste** Treffer (nicht `max()` über mehrere). Grössere Toleranz + `max()` liefert sonst bei Grenzfällen einen viel zu hohen Wert aus einer Nachbarzone.
- `identify_population` + `parse_population`: **nächstgelegene** Hektarzelle, nicht die Summe.


In [ ]:

def safe_num(x, default=None) -> Optional[float]:
    try:
        if x is None:
            return default
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default


def safe_int(x, default=None) -> Optional[int]:
    v = safe_num(x, None)
    if v is None:
        return default
    try:
        return int(round(v))
    except Exception:
        return default


def _request(url: str, *, params=None, timeout=15, session: requests.Session = None):
    sess = session or requests
    for attempt in range(2):
        try:
            r = sess.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r
            return None
        except requests.exceptions.ConnectionError:
            if attempt == 0:
                time.sleep(2)
        except Exception:
            return None
    return None


def _get_json(url: str, params: dict, timeout=15, session: requests.Session = None):
    r = _request(url, params=params, timeout=timeout, session=session)
    if not r:
        return None
    try:
        return r.json()
    except Exception:
        return None


In [ ]:

def geocode(address: str, session: requests.Session = None) -> Optional[Tuple[float, float]]:
    d = _get_json(API_SEARCH_URL, {
        "searchText": address,
        "type": "locations",
        "origins": "address",
        "sr": 2056,
        "limit": 1,
    }, timeout=15, session=session)

    if not d or not d.get("results"):
        return None

    attrs = d["results"][0].get("attrs", {}) or {}
    east  = safe_num(attrs.get("y"))   # y = East in LV95
    north = safe_num(attrs.get("x"))   # x = North in LV95

    if east is None or north is None:
        return None
    return east, north


def get_elevation(east: float, north: float, session: requests.Session = None) -> Optional[float]:
    d = _get_json(API_HEIGHT_URL, {"easting": east, "northing": north},
                  timeout=15, session=session)
    if not d:
        return None
    return safe_num(d.get("height"))


In [ ]:

def identify(east: float, north: float, session: requests.Session = None) -> dict:
    # tolerance=1: nimm NUR die Features, die den Punkt tatsächlich enthalten.
    # Grosse Toleranz würde Nachbarzonen/Nachbardächer mitliefern.
    return _get_json(
        API_IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": IDENTIFY_LAYERS,
            "tolerance": 1,
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "100,100,96",
            "mapExtent": f"{east-10},{north-10},{east+10},{north+10}",
        },
        timeout=20, session=session,
    ) or {"results": []}


def parse_identify(results: list) -> Dict[str, Any]:
    # identify liefert Features nach Nähe sortiert -> pro Layer der ERSTE Treffer.
    #
    # solar_class = int 1..5 gemäss BFE-Domäne EIGNUNG_DACH:
    #   1 = Gering, 2 = Mittel, 3 = Gut, 4 = Sehr gut, 5 = Top (hervorragend)
    out = {"oev_score": None, "solar_class": None}
    oev_done   = False
    solar_done = False

    for item in results:
        layer = item.get("layerBodId", "")
        attr  = item.get("attributes", {}) or {}

        if not oev_done and layer == "ch.are.erreichbarkeit-oev":
            out["oev_score"] = safe_num(attr.get("oev_erreichb_ewap"))
            oev_done = True

        elif not solar_done and layer == "ch.bfe.solarenergie-eignung-daecher":
            out["solar_class"] = safe_int(attr.get("klasse"))
            solar_done = True

        if oev_done and solar_done:
            break
    return out


def identify_population(east: float, north: float, session: requests.Session = None) -> dict:
    return _get_json(
        API_IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": POP_LAYER,
            "tolerance": 1,
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "100,100,96",
            "mapExtent": f"{east-10},{north-10},{east+10},{north+10}",
        },
        timeout=15, session=session,
    ) or {"results": []}


def parse_population(results: list) -> Optional[int]:
    # Nimmt NUR den ersten Treffer (= nächstgelegene Hektarzelle).
    # NICHT summieren, sonst werden an Zellgrenzen Nachbarzellen doppelt gezählt.
    for item in results:
        attr = item.get("attributes", {}) or {}
        number_val = safe_int(attr.get("number"), None)
        year_val   = attr.get("i_year")
        if number_val is None:
            continue
        if year_val is None or year_val == 2024:
            return number_val
    return None


In [ ]:

def enrich_address(address: str, session: requests.Session) -> Dict[str, Any]:
    coords = geocode(address, session=session)
    if not coords:
        return {"status": "geocode_failed", "address": address}
    east, north = coords

    parsed     = parse_identify(identify(east, north, session=session).get("results") or [])
    population = parse_population(identify_population(east, north, session=session).get("results") or [])
    elevation  = get_elevation(east, north, session=session)

    return {
        "status":          "ok",
        "address":         address,
        "lv95_east":       east,
        "lv95_north":      north,
        "oev_score":       parsed["oev_score"],
        "solar_class":     parsed["solar_class"],   # 1..5
        "elevation_m":     elevation,
        "population":      population,
        # noch nicht implementiert - bewusst None:
        "pt_distance_m":   None,
        "dist_autobahn_m": None,
        "dist_school_m":   None,
    }


## Einzeltest – eine Adresse vorher probieren

Hier kannst du **eine** Adresse aus `listing_df` durch die volle Chain schicken, bevor du den grossen Loop startest. Gut geeignet, um Layer-Antworten zu inspizieren.


In [ ]:

# wähle eine Testadresse: entweder aus der DB oder hart reingetippt
test_address = listing_df["address_norm"].iloc[0]
# test_address = "Bundesplatz 3, 3003 Bern"

print("Teste:", test_address)

with requests.Session() as _s:
    _s.headers.update({"User-Agent": "swisstopo-enrich-test/1.0"})
    _result = enrich_address(test_address, _s)

display(pd.DataFrame([_result]))


In [ ]:

# Optional: Roh-Antworten inspizieren (für Debugging einzelner Layer)
with requests.Session() as _s:
    _s.headers.update({"User-Agent": "swisstopo-enrich-test/1.0"})
    _coords = geocode(test_address, session=_s)
    print("coords:", _coords)

    if _coords:
        _e, _n = _coords

        _ident = identify(_e, _n, session=_s)
        print(f"\nidentify ÖV+Solar: {len(_ident.get('results') or [])} Treffer")
        for _it in (_ident.get("results") or [])[:5]:
            _a = _it.get("attributes") or {}
            _layer = _it.get("layerBodId")
            if _layer == "ch.are.erreichbarkeit-oev":
                print(f"   ÖV    -> oev_erreichb_ewap={_a.get('oev_erreichb_ewap')}")
            elif _layer == "ch.bfe.solarenergie-eignung-daecher":
                print(f"   Solar -> klasse={_a.get('klasse')} "
                      f"mstrahlung={_a.get('mstrahlung')} kWh/m²/a "
                      f"gstrahlung={_a.get('gstrahlung')} kWh/a "
                      f"flaeche={_a.get('flaeche')} m²")
            else:
                print(f"   {_layer}: {list(_a.keys())[:6]}")

        _pop = identify_population(_e, _n, session=_s)
        print(f"\nidentify Bevölkerung: {len(_pop.get('results') or [])} Zellen")
        for _it in (_pop.get("results") or []):
            _a = _it.get("attributes", {}) or {}
            print(f"   cell reli={_a.get('reli')} year={_a.get('i_year')} pop={_a.get('number')}")


## Lookup für alle eindeutigen Adressen

Mehrere Listings können dieselbe Adresse haben → erst dedupen, dann einen API-Call pro eindeutiger Adresse. Anschliessend wird die Anreicherung zurück auf alle `listing_id`s gemappt.


In [ ]:

unique_addresses = (
    listing_df[["address_norm"]]
    .drop_duplicates()
    .rename(columns={"address_norm": "address"})
    .reset_index(drop=True)
)

print("Eindeutige Adressen:", len(unique_addresses))

lookup_rows = []
errors = []

with requests.Session() as session:
    session.headers.update({"User-Agent": "swisstopo-enrich-db-sync/1.0"})

    for address in tqdm(unique_addresses["address"], desc="Adresse → Swisstopo"):
        try:
            row = enrich_address(address, session)
            lookup_rows.append(row)
        except Exception as e:
            lookup_rows.append({
                "status": f"error: {type(e).__name__}: {str(e)[:180]}",
                "address": address,
            })
            errors.append(address)

        time.sleep(DELAY)

lookup_df = pd.DataFrame(lookup_rows)
print("Lookup fertig:", len(lookup_df))
print("davon ok      :", (lookup_df["status"] == "ok").sum())
print("Fehler         :", len(errors))
display(lookup_df.head(10))


## Mapping listing_id → Anreicherung

Jede erfolgreich angereicherte Adresse wird wieder auf alle `listing_id`s verteilt, die diese Adresse haben.


In [ ]:

good_lookup = lookup_df[lookup_df["status"] == "ok"].drop(columns=["status"]).copy()

merged = listing_df.merge(
    good_lookup,
    left_on="address_norm",
    right_on="address",
    how="inner",
    suffixes=("_listing", "_lookup"),
)

print("Listings mit Treffer:", len(merged))
display(merged.head(10))


## Für Insert vorbereiten

Es werden nur Spalten verwendet, die tatsächlich in `swisstopo_details` existieren. Die Solar-Eignung wird als Integer-Klasse `solar_class` (1–5) gespeichert. Falls die Spalte noch fehlt, einmalig hinzufügen:

```sql
ALTER TABLE public.swisstopo_details
  ADD COLUMN IF NOT EXISTS solar_class int;
```

Gemäss BFE-Doku bedeutet: 1=Gering, 2=Mittel, 3=Gut, 4=Sehr gut, 5=Top.

Fehlt die Spalte, wird sie beim Insert übersprungen – der Upsert schreibt dann nur das, was in der Tabelle existiert.


In [ ]:

with engine.connect() as conn:
    col_query = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
        ORDER BY ordinal_position
    """)
    target_columns = [r[0] for r in conn.execute(col_query, {"schema": SCHEMA, "table": TARGET_TABLE})]

print("Spalten in swisstopo_details:")
print(target_columns)


In [ ]:

# Zielspalten, die wir befüllen können
candidate_cols = [
    "listing_id",
    "oev_score",
    "solar_class",
    "elevation_m", "population",
    "pt_distance_m", "dist_autobahn_m", "dist_school_m",
    "lv95_east", "lv95_north",
]

insertable_cols = [c for c in candidate_cols if c in target_columns]
missing_in_target = [c for c in candidate_cols if c not in target_columns]
if missing_in_target:
    print("⚠ Diese Kandidatenspalten fehlen in der Zieltabelle und werden weggelassen:",
          missing_in_target)

db_df = merged[insertable_cols].copy()
db_df = db_df.drop_duplicates(subset=["listing_id"]).reset_index(drop=True)

# numerische Float-Spalten
for c in ["oev_score", "elevation_m", "lv95_east", "lv95_north",
          "pt_distance_m", "dist_autobahn_m", "dist_school_m"]:
    if c in db_df.columns:
        db_df[c] = pd.to_numeric(db_df[c], errors="coerce")

# Integer-Spalten (nullable)
for c in ["population", "listing_id", "solar_class"]:
    if c in db_df.columns:
        db_df[c] = pd.to_numeric(db_df[c], errors="coerce").astype("Int64")

print("Zeilen bereit für DB:", len(db_df))
print("Spalten für Insert   :", insertable_cols)
display(db_df.head(10))


## Upsert in `swisstopo_details`

`listing_id` ist der natürliche Schlüssel. Der Upsert läuft idempotent – du kannst den Loop und diese Zelle gefahrlos mehrfach ausführen.

Falls `listing_id` **kein Unique/Primary Key** ist, schlägt `ON CONFLICT (listing_id)` fehl. In dem Fall unten die Diagnose-Zelle ausführen.


In [ ]:

INT_COLS = {"listing_id", "population", "solar_class"}

def upsert_swisstopo(df: pd.DataFrame, engine, schema: str, table: str) -> int:
    if df.empty:
        print("Nichts zu schreiben.")
        return 0

    cols = list(df.columns)
    placeholders = ", ".join([f":{c}" for c in cols])
    quoted_cols  = ", ".join(cols)

    update_cols = [c for c in cols if c != "listing_id"]
    set_clause  = ", ".join([f"{c} = EXCLUDED.{c}" for c in update_cols])

    sql = text(f"""
        INSERT INTO {schema}.{table} ({quoted_cols})
        VALUES ({placeholders})
        ON CONFLICT (listing_id) DO UPDATE
        SET {set_clause}
    """)

    payload = []
    for rec in df.to_dict("records"):
        clean = {}
        for k, v in rec.items():
            if pd.isna(v):
                clean[k] = None
            elif k in INT_COLS:
                clean[k] = int(v)
            else:
                clean[k] = v
        payload.append(clean)

    with engine.begin() as conn:
        conn.execute(sql, payload)

    return len(payload)


written = upsert_swisstopo(db_df, engine, SCHEMA, TARGET_TABLE)
print(f"✅ In {TARGET_TABLE} geschrieben/aktualisiert: {written}")


## Kontrolle in der DB

In [5]:

summary_sql = text(f"""
    SELECT
        COUNT(*)                                         AS rows_total,
        COUNT(*) FILTER (WHERE oev_score   IS NULL)      AS oev_null,
        COUNT(*) FILTER (WHERE solar_class IS NULL)      AS solar_null,
        COUNT(*) FILTER (WHERE elevation_m IS NULL)      AS elev_null,
        COUNT(*) FILTER (WHERE population  IS NULL)      AS pop_null,
        MIN(elevation_m)                                 AS elev_min,
        MAX(elevation_m)                                 AS elev_max,
        MIN(oev_score)                                   AS oev_min,
        MAX(oev_score)                                   AS oev_max,
        MIN(solar_class)                                 AS solar_class_min,
        MAX(solar_class)                                 AS solar_class_max
    FROM {SCHEMA}.{TARGET_TABLE}
""")

summary_df = pd.read_sql(summary_sql, engine)
display(summary_df)


,rows_total,oev_null,solar_null,elev_null,pop_null,elev_min,elev_max,oev_min,oev_max,solar_class_min,solar_class_max
0,9666,27,1581,0,556,195.8,2873.4,0.0,144795.0,1.0,481331.0


In [ ]:

sample_sql = text(f"""
    SELECT s.listing_id, l.address,
           s.lv95_east, s.lv95_north,
           s.oev_score, s.solar_class,
           s.elevation_m, s.population
    FROM {SCHEMA}.{TARGET_TABLE} s
    JOIN {SCHEMA}.{LISTING_DETAILS_TABLE} l USING (listing_id)
    ORDER BY random()
    LIMIT 10
""")

display(pd.read_sql(sample_sql, engine))


## Optional: Diagnose, falls `ON CONFLICT (listing_id)` fehlschlägt

In [ ]:

diag_sql = text("""
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
  ON tc.constraint_name = kcu.constraint_name
 AND tc.table_schema = kcu.table_schema
WHERE tc.table_schema = :schema
  AND tc.table_name = :table
ORDER BY tc.constraint_type, tc.constraint_name, kcu.ordinal_position
""")

with engine.connect() as conn:
    constraints_df = pd.read_sql(diag_sql, conn, params={"schema": SCHEMA, "table": TARGET_TABLE})

display(constraints_df)
